# Mochiichao character LoRAs — Wispkin + Dorowani (musubi-tuner)

Trains the first two Mochiichao character LoRAs from canon art only, with
[kohya-ss/musubi-tuner](https://github.com/kohya-ss/musubi-tuner) (Apache-2.0).

**Verified base-model note (2026-10-28):** musubi-tuner @ main `f8a1b03` has **no `ti2v-5B`
task** in `WAN_CONFIGS` — Wan2.2-TI2V-5B cannot be trained with musubi-tuner yet. This kernel
therefore trains on **Wan2.1 T2V-1.3B** (`--task t2v-1.3B`, the documented low-VRAM example in
docs/wan.md), which fits a Kaggle T4. The mounted `mochiichao-wan22-core` dataset (the TI2V-5B
scene model) is inventoried in cell 2 for the record; it is not the training base.
Outputs land in `/kaggle/working/lora-output/` with sha256 + sizes.


In [ ]:
import glob, os, shutil, subprocess, sys, json, hashlib
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout[:800])
# --- inventory the mounted Wan core dataset (record only) ---
core = glob.glob('/kaggle/input/**/wan2.2_ti2v_5B_fp16.safetensors', recursive=True)
print('wan22-core TI2V-5B file:', core)
# --- stage our datasets + configs into the canonical working layout ---
# NOTE: this account's Kaggle CLI drops dataset SUBFOLDERS on upload (verified twice:
# v1 nested and v2 flat-folder mounts contained only root files), so the dataset ships
# all files at root and the per-character directories are rebuilt here.
hit = glob.glob('/kaggle/input/**/train_wispkin.toml', recursive=True)
assert hit, 'mochiichao-lora-datasets not mounted (train_wispkin.toml not found)'
src = os.path.dirname(hit[0])
print('lora dataset root:', src, sorted(os.listdir(src))[:8], '... total', len(os.listdir(src)))
os.makedirs('/kaggle/working/data', exist_ok=True); os.makedirs('/kaggle/working/configs', exist_ok=True)
for ch in ['wispkin','dorowani']:
    d = f'/kaggle/working/data/{ch}'; os.makedirs(d, exist_ok=True)
    n = 0
    for f in glob.glob(os.path.join(src, ch + '_*.png')) + glob.glob(os.path.join(src, ch + '_*.txt')):
        shutil.copy(f, os.path.join(d, os.path.basename(f))); n += 1
    print(ch, len(glob.glob(d + '/*.png')), 'images,', len(glob.glob(d + '/*.txt')), 'captions')
    assert len(glob.glob(d + '/*.png')) >= 20, ch + ' dataset incomplete'
for f in glob.glob(os.path.join(src,'*.toml')):
    shutil.copy(f, '/kaggle/working/configs/'+os.path.basename(f))
print(sorted(os.listdir('/kaggle/working/configs')))


In [ ]:
# --- musubi-tuner: clone pinned commit, install, verify task support at source ---
!git clone -q https://github.com/kohya-ss/musubi-tuner.git /kaggle/working/musubi-tuner
%cd /kaggle/working/musubi-tuner
!git checkout -q f8a1b03794a49239a3539015075f5123d6c07d66
!pip install -q -e .
import re
cfg = open('src/musubi_tuner/wan/configs/__init__.py').read()
tasks = re.findall(r'"([A-Za-z0-9.\-]+)":', cfg.split('WAN_CONFIGS = {')[1].split('}')[0])
print('WAN_CONFIGS tasks:', tasks)
print('ti2v-5B supported:', 'ti2v-5B' in tasks, ' <- expected False on this commit; TI2V-5B training is NOT possible with musubi-tuner yet')
assert 't2v-1.3B' in tasks


In [ ]:
# --- base model files (paths per musubi docs/wan.md) ---
from huggingface_hub import hf_hub_download
os.makedirs('/kaggle/working/models', exist_ok=True)
def get(repo, fn, dest):
    p = hf_hub_download(repo_id=repo, filename=fn)
    shutil.copy(p, dest); print(dest, os.path.getsize(dest)); return dest
vae = get('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/vae/wan_2.1_vae.safetensors', '/kaggle/working/models/wan_2.1_vae.safetensors')
t5  = get('Wan-AI/Wan2.1-I2V-14B-720P', 'models_t5_umt5-xxl-enc-bf16.pth', '/kaggle/working/models/models_t5_umt5-xxl-enc-bf16.pth')
dit = None; mp = 'fp16'
for fn, m in [('split_files/diffusion_models/wan2.1_t2v_1.3B_fp16.safetensors','fp16'),
              ('split_files/diffusion_models/wan2.1_t2v_1.3B_bf16.safetensors','bf16')]:
    try:
        dit = get('Comfy-Org/Wan_2.1_ComfyUI_repackaged', fn, '/kaggle/working/models/'+os.path.basename(fn)); mp = m; break
    except Exception as e:
        print('not found:', fn, type(e).__name__)
assert dit, 'no Wan2.1 T2V-1.3B DiT found in Comfy-Org repackaged repo'
print('DiT:', dit, 'mixed_precision:', mp)
# point the train TOMLs at the DiT that actually landed
for ch in ['wispkin','dorowani']:
    p = f'/kaggle/working/configs/train_{ch}.toml'; s = open(p).read()
    s = s.replace('/kaggle/working/models/wan2.1_t2v_1.3B_fp16.safetensors', dit)
    if mp == 'bf16': s = s.replace('mixed_precision = "fp16"', 'mixed_precision = "bf16"')
    open(p,'w').write(s)
MIXED = mp


In [ ]:
# --- train both characters: cache latents -> cache T5 outputs -> train (musubi 3-step, docs/wan.md) ---
import subprocess, os
MT = '/kaggle/working/musubi-tuner/src/musubi_tuner'
def run(cmd):
    print('+', ' '.join(cmd)); r = subprocess.run(cmd, cwd='/kaggle/working/musubi-tuner')
    print('rc=', r.returncode); return r.returncode
results = {}
for ch in ['wispkin','dorowani']:
    dc = f'/kaggle/working/configs/dataset_{ch}.toml'
    rc = run([sys.executable, f'{MT}/wan_cache_latents.py', '--dataset_config', dc, '--vae', '/kaggle/working/models/wan_2.1_vae.safetensors'])
    if rc == 0:
        rc = run([sys.executable, f'{MT}/wan_cache_text_encoder_outputs.py', '--dataset_config', dc,
                  '--t5', '/kaggle/working/models/models_t5_umt5-xxl-enc-bf16.pth', '--batch_size', '8', '--fp8_t5'])
    if rc == 0:
        rc = run(['accelerate', 'launch', '--num_cpu_threads_per_process', '1', '--mixed_precision', MIXED,
                  f'{MT}/wan_train_network.py', '--config_file', f'/kaggle/working/configs/train_{ch}.toml'])
    results[ch] = rc
print('RESULTS', results)


In [ ]:
# --- collect outputs with hashes ---
os.makedirs('/kaggle/working/lora-output', exist_ok=True)
man = {}
for ch in ['wispkin','dorowani']:
    for p in sorted(glob.glob(f'/kaggle/working/out/{ch}/*.safetensors')):
        dest = '/kaggle/working/lora-output/'+os.path.basename(p)
        shutil.copy(p, dest)
        man[os.path.basename(p)] = {'bytes': os.path.getsize(dest),
            'sha256': hashlib.sha256(open(dest,'rb').read()).hexdigest(), 'character': ch,
            'base_model': 'Wan2.1 T2V-1.3B (musubi-tuner task t2v-1.3B)', 'train_rc': results.get(ch)}
json.dump(man, open('/kaggle/working/lora-output/lora-files.json','w'), indent=1)
print(json.dumps(man, indent=1))
assert man, 'NO LoRA FILES PRODUCED - see training logs above'
